# Single-layer debiasing — activation extraction

For each possible debias layer L (0..23):
1. Trains a CAV on raw activations at layer L.
2. Computes post-debiasing activations at layer L (from raw, no model rerun needed).
3. Runs CLIP with a debiasing hook at layer L — captures activations at layers L+1..23
   and the final 768-dim model output (after `post_layernorm` + `visual_projection`).
4. Saves everything to structured parquets + info.json.

**Change `CONCEPT` and `METHODS`** — everything else follows automatically.

Prerequisite: `02_get_activations.ipynb` must have been run (raw activations must exist).

Output structure:
```
data/activations/debiased/single/{CONCEPT}/{method}/
  from_layer_{LL}/
    train/
      layer_{LL}.parquet     # post-debiasing at LL (projected from raw)
      layer_{LL+1}.parquet   # propagated through network after debiasing at LL
      ...
      layer_23.parquet
      model_output.parquet   # 768-dim CLIP image embedding after debiasing at LL
    test/  (same)
    info.json
  cavs.csv                   # one row per layer: CAV vector + accuracy metrics
```

Parquet schema: `filename (str), 0..1023 (float16)` for layers; `filename (str), 0..767 (float16)` for model_output.

In [1]:
import json
import os
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from dotenv import load_dotenv
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from transformers import AutoModel, AutoImageProcessor
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
load_dotenv()

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'pyproject.toml').exists():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from software.torch_lr import TorchLR

In [2]:
import random, numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [3]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
CONCEPT        = 'eyeglasses'
NUM_LAYERS     = 24
GPU_BATCH_SIZE = 64
NUM_WORKERS    = 0
MODEL_ID       = 'openai/clip-vit-large-patch14'
PARQUET_COMPRESSION = 'snappy'
METHODS        = ['diff_means', 'lr', 'pclarc']

# Solver for LR-based CAV (method='lr'):
#   'torch_lr' — TorchLR (LBFGS on GPU, stable, recommended)
#   'sgd'      — SGDClassifier from sklearn (stochastic gradient descent)
SOLVER   = 'torch_lr'
SOLVER_C = 0.1

METADATA_PATH  = ROOT / 'data' / 'metadata.csv'
IMAGES_DIR     = ROOT / 'data' / 'images'
RAW_ACT_DIR    = ROOT / 'data' / 'activations' / 'raw'
DATA_OUT       = ROOT / 'data' / 'activations' / 'debiased' / 'single' / CONCEPT

assert METADATA_PATH.exists(), f'Missing: {METADATA_PATH}'
assert IMAGES_DIR.exists(),    f'Missing: {IMAGES_DIR}'
assert RAW_ACT_DIR.exists(),   f'Run 02_get_activations.ipynb first. Missing: {RAW_ACT_DIR}'

df_meta  = pd.read_csv(METADATA_PATH)
assert CONCEPT in df_meta.columns, f'Column "{CONCEPT}" not in metadata.csv'
df_train = df_meta[df_meta['split'] == 'train'][['filename', CONCEPT]].reset_index(drop=True)
df_test  = df_meta[df_meta['split'] == 'test'][['filename',  CONCEPT]].reset_index(drop=True)

print(f'Concept  : {CONCEPT}')
print(f'Solver   : {SOLVER} (C={SOLVER_C})')
print(f'Train    : {len(df_train)} | Test: {len(df_test)}')
print(f'Data out : {DATA_OUT}')

Concept  : eyeglasses
Solver   : torch_lr (C=0.1)
Train    : 4642 | Test: 715
Data out : /workspace/WB2/data/activations/debiased/single/eyeglasses


In [4]:
HF_TOKEN = os.getenv('HF_TOKEN')
device   = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype    = torch.bfloat16 if device == 'cuda' else torch.float32

processor = AutoImageProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModel.from_pretrained(
    MODEL_ID, torch_dtype=dtype, low_cpu_mem_usage=True, token=HF_TOKEN,
).to(device).eval()
print(f'Model: {MODEL_ID} | {device} | {dtype}')


class CelebADataset(Dataset):
    def __init__(self, df, images_dir, processor):
        self.df = df.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.processor  = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.images_dir / row['filename']).convert('RGB')
        px  = self.processor(images=img, return_tensors='pt').pixel_values.squeeze(0)
        return px, row['filename'], int(row[CONCEPT])


def make_loader(df):
    return DataLoader(
        CelebADataset(df, IMAGES_DIR, processor),
        batch_size=GPU_BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=(device == 'cuda'),
        persistent_workers=(NUM_WORKERS > 0),
    )


loader_train = make_loader(df_train)
loader_test  = make_loader(df_test)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

Model: openai/clip-vit-large-patch14 | cuda | torch.bfloat16


In [5]:
def make_cav_clf():
    """LR classifier for CAV training (no CV — C fixed by SOLVER_C)."""
    if SOLVER == 'torch_lr':
        return TorchLR(C=SOLVER_C, max_iter=500, random_state=42)
    return SGDClassifier(
        loss='log_loss', penalty='l2',
        alpha=1.0 / (2.0 * SOLVER_C),
        max_iter=1000, tol=1e-4, random_state=42,
    )


def load_raw_layer_by_split(split_label, layer_idx):
    """Load raw activations at layer_idx for given split. Returns (X float32, filenames, y)."""
    split_df = df_train if split_label == 'train' else df_test
    df_raw = pd.read_parquet(RAW_ACT_DIR / split_label / f'layer_{layer_idx:02d}.parquet')
    df_raw = df_raw.merge(split_df, on='filename', how='left')
    feat_cols = [c for c in df_raw.columns if c not in ('filename', CONCEPT)]
    return df_raw[feat_cols].values.astype(np.float32), df_raw['filename'].tolist(), df_raw[CONCEPT].values.astype(int)


def compute_diff_means(X_tr, y_tr):
    X = X_tr.astype(np.float64)
    diff = X[y_tr == 1].mean(axis=0) - X[y_tr == 0].mean(axis=0)
    return (diff / np.linalg.norm(diff)).astype(np.float32)


def train_cav(X_tr, y_tr, method):
    """Returns (cav, meta_dict).

    For 'lr', the LR intercept is rescaled by ||w|| so the discriminant
    `X @ cav + lr_intercept_scaled > 0` matches the full LR `clf.predict(X)`.
    For 'diff_means' / 'pclarc', threshold is the midpoint between class
    projection means.
    """
    threshold = target_val = best_C = None
    lr_intercept_scaled = None  # only set for method == 'lr'

    if method == 'lr':
        clf = make_cav_clf()
        clf.fit(X_tr, y_tr)
        w = clf.coef_[0].astype(np.float64)
        w_norm = float(np.linalg.norm(w)) + 1e-12
        cav = (w / w_norm).astype(np.float32)
        # LR decision: w·x + b > 0  <=>  cav·x + b/||w|| > 0
        b_raw = float(np.atleast_1d(clf.intercept_)[0]) if hasattr(clf, 'intercept_') else 0.0
        lr_intercept_scaled = b_raw / w_norm
        best_C = float(getattr(clf, 'C_', [SOLVER_C])[0]) if hasattr(clf, 'C_') else SOLVER_C
        # Train accuracy with the rescaled-intercept rule (matches clf.predict)
        proj = X_tr.astype(np.float64) @ cav.astype(np.float64)
        acc_tr = accuracy_score(y_tr, (proj + lr_intercept_scaled > 0).astype(int))
    else:
        cav = compute_diff_means(X_tr, y_tr)
        proj = X_tr.astype(np.float64) @ cav.astype(np.float64)
        threshold = float((proj[y_tr == 1].mean() + proj[y_tr == 0].mean()) / 2)
        acc_tr = accuracy_score(y_tr, (proj > threshold).astype(int))

    if method == 'pclarc':
        target_val = float((X_tr.astype(np.float64)[y_tr == 0] @ cav.astype(np.float64)).mean())

    return cav, {'threshold': threshold, 'target_val': target_val, 'lr_best_C': best_C,
                 'lr_intercept': lr_intercept_scaled, 'train_acc': acc_tr}


def eval_cav_accuracy(X, y, cav, method, meta):
    proj = X.astype(np.float64) @ cav.astype(np.float64)
    if method == 'lr':
        # #1 — use the saved intercept so the test rule matches LR's trained boundary
        b = meta.get('lr_intercept', 0.0) or 0.0
        return accuracy_score(y, (proj + b > 0).astype(int))
    # diff_means / pclarc: midpoint threshold (already saved as meta['threshold'])
    return accuracy_score(y, (proj > meta['threshold']).astype(int))


def apply_debiasing(X, cav, method, meta):
    X64 = X.astype(np.float64)
    c64 = cav.astype(np.float64)
    if method == 'pclarc':
        t = meta['target_val']
        return (X64 + np.outer(t - X64 @ c64, c64)).astype(np.float32)
    return (X64 - np.outer(X64 @ c64, c64)).astype(np.float32)


def _make_debias_hook(cav_np, method, target_val=None):
    cav_t = torch.from_numpy(cav_np).float()

    def hook(module, input, output):
        is_tuple = isinstance(output, tuple)
        hidden   = output[0] if is_tuple else output
        cls      = hidden[:, 0, :].float()
        cav_d    = cav_t.to(cls.device)
        proj     = cls @ cav_d
        if method in ('diff_means', 'lr'):
            cls_new = cls - proj.unsqueeze(1) * cav_d
        else:
            t       = torch.tensor(target_val, dtype=torch.float32, device=cls.device)
            cls_new = cls + (t - proj).unsqueeze(1) * cav_d
        h_new = hidden.clone()
        h_new[:, 0, :] = cls_new.to(hidden.dtype)
        return (h_new,) + output[1:] if is_tuple else h_new

    return hook


def run_with_single_hook(loader, debias_layer, cav_np, method, target_val=None):
    """Run CLIP with a debiasing hook at debias_layer.
    Captures CLS-token activations at layers debias_layer+1..23 and the final 768-dim model output.
    Returns (layer_acts_dict, filenames, y, outputs_768).
    """
    encoder_layers = model.vision_model.encoder.layers
    handles        = []
    acts_bufs      = {l: [] for l in range(debias_layer + 1, NUM_LAYERS)}
    outputs_buf    = []
    fn_all, y_all  = [], []

    handles.append(
        encoder_layers[debias_layer].register_forward_hook(
            _make_debias_hook(cav_np, method, target_val)
        )
    )

    for l in range(debias_layer + 1, NUM_LAYERS):
        def make_capture(buf):
            def hook(module, input, output):
                hidden = output[0] if isinstance(output, tuple) else output
                buf.append(hidden[:, 0, :].detach().float().cpu().numpy())
            return hook
        handles.append(encoder_layers[l].register_forward_hook(make_capture(acts_bufs[l])))

    def capture_model_output(module, input, output):
        outputs_buf.append(output.detach().float().cpu().numpy())

    handles.append(model.visual_projection.register_forward_hook(capture_model_output))

    try:
        with torch.no_grad():
            for pixels, fnames, labels in loader:
                model.get_image_features(pixel_values=pixels.to(device, dtype=dtype))
                fn_all.extend(fnames)
                y_all.extend(labels.tolist())
    finally:
        for h in handles:
            h.remove()

    result       = {l: np.concatenate(acts_bufs[l], axis=0) for l in range(debias_layer + 1, NUM_LAYERS)}
    outputs_768  = np.concatenate(outputs_buf, axis=0)
    return result, list(fn_all), np.array(y_all, dtype=int), outputs_768


def save_parquet(X, filenames, out_path):
    df = pd.DataFrame(X.astype(np.float16))
    df.columns = df.columns.astype(str)
    df.insert(0, 'filename', filenames)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out_path, compression=PARQUET_COMPRESSION, index=False)

In [6]:
for method in METHODS:
    cavs_path = DATA_OUT / method / 'cavs.csv'
    if cavs_path.exists():
        print(f'[{method}] cavs.csv exists — skipping.')
        continue

    print(f'\n=== Method: {method} ===')
    cav_records = []

    for debias_layer in tqdm(range(NUM_LAYERS), desc=method):
        out_base = DATA_OUT / method / f'from_layer_{debias_layer:02d}'

        # ── 1. Load raw activations at debias_layer, train CAV ──
        X_tr, fn_tr, y_tr = load_raw_layer_by_split('train', debias_layer)
        X_te, fn_te, y_te = load_raw_layer_by_split('test',  debias_layer)

        cav, meta = train_cav(X_tr, y_tr, method)
        acc_te = eval_cav_accuracy(X_te, y_te, cav, method, meta)

        cav_records.append({
            'layer_id':     debias_layer,
            'train_acc':    meta['train_acc'],
            'test_acc':     acc_te,
            'threshold':    meta['threshold'],
            'target_val':   meta['target_val'],
            'lr_best_C':    meta['lr_best_C'],
            'lr_intercept': meta.get('lr_intercept'),
            **{f'cav_{i}': float(cav[i]) for i in range(len(cav))},
        })

        # ── 2. Save post-debiasing at debias_layer (from raw, no rerun needed) ──
        X_tr_post = apply_debiasing(X_tr, cav, method, meta)
        X_te_post = apply_debiasing(X_te, cav, method, meta)

        save_parquet(X_tr_post, fn_tr, out_base / 'train' / f'layer_{debias_layer:02d}.parquet')
        save_parquet(X_te_post, fn_te, out_base / 'test'  / f'layer_{debias_layer:02d}.parquet')

        # ── 3. Run CLIP with hook — capture downstream layers + final model output ──
        target_val = meta['target_val']

        downstream_tr, fn_tr2, _, outputs_tr = run_with_single_hook(
            loader_train, debias_layer, cav, method, target_val
        )
        downstream_te, fn_te2, _, outputs_te = run_with_single_hook(
            loader_test, debias_layer, cav, method, target_val
        )

        for l in range(debias_layer + 1, NUM_LAYERS):
            save_parquet(downstream_tr[l], fn_tr2, out_base / 'train' / f'layer_{l:02d}.parquet')
            save_parquet(downstream_te[l], fn_te2, out_base / 'test'  / f'layer_{l:02d}.parquet')

        save_parquet(outputs_tr, fn_tr2, out_base / 'train' / 'model_output.parquet')
        save_parquet(outputs_te, fn_te2, out_base / 'test'  / 'model_output.parquet')

        # ── 4. info.json ──
        layers_saved = list(range(debias_layer, NUM_LAYERS))
        info = {
            'concept':              CONCEPT,
            'method':               method,
            'debias_type':          'single',
            'debias_layer':         debias_layer,
            'layers_saved':         layers_saved,
            'model_output_saved':   True,
            'parquet_schema':       'filename (str), 0..1023 (float16)',
            'model_output_schema':  'filename (str), 0..767 (float16)',
            'model_id':             MODEL_ID,
            'n_features':           1024,
            'n_output_features':    768,
            'created_at':           datetime.now(timezone.utc).isoformat(),
            'description': (
                f"Single-layer debiasing of concept '{CONCEPT}' at layer {debias_layer} "
                f"using {method}. layer_{debias_layer:02d}.parquet contains activations "
                f"after orthogonal projection at that layer (computed from raw). "
                f"Subsequent layers reflect this debiasing propagated through the network. "
                f"model_output.parquet contains the 768-dim CLIP image embedding "
                f"(post_layernorm + visual_projection, before L2 norm) with the debiasing applied. "
                f"Layers 0..{debias_layer - 1} are unmodified (use raw activations). "
                f"Labels: join on 'filename' with data/metadata.csv."
            ),
        }
        with open(out_base / 'info.json', 'w') as f:
            json.dump(info, f, indent=2)

    # ── 5. Save CAV records ──
    (DATA_OUT / method).mkdir(parents=True, exist_ok=True)
    pd.DataFrame(cav_records).to_csv(DATA_OUT / method / 'cavs.csv', index=False)
    print(f'  Saved cavs.csv for method={method}')

print('\nDone.')


=== Method: diff_means ===


diff_means:   0%|          | 0/24 [00:00<?, ?it/s]

  Saved cavs.csv for method=diff_means

=== Method: lr ===


lr:   0%|          | 0/24 [00:00<?, ?it/s]

  Saved cavs.csv for method=lr

=== Method: pclarc ===


pclarc:   0%|          | 0/24 [00:00<?, ?it/s]

  Saved cavs.csv for method=pclarc

Done.
